# Retrieval Augmented Generation (RAG)

$\bullet$ Retrieval-Augmented Generation (RAG) is a framework that enhances large language models (LLMs) by combining a non-generative retrieval mechanism with a generative model, enabling the system to produce responses grounded in externally retrieved information.  
$\bullet$ Here are two behaviours that are often observed as problematic when interacting with large language models.  
$\qquad \blacksquare$ No source  
$\qquad \blacksquare$ Out of date  
$\bullet$ RAG approach improves factual accuracy and reduces hallucinations by grounding the generated responses in real, up-to-date information.  
$\bullet$ Typically, a RAG system consists of two main components:  
$\qquad \blacksquare$ Retriever  
$\qquad \blacksquare$ Generator  
$\bullet$ Formally, let $x$ be the input query and $D = \{d_1, d_2, \dots, d_n\}$ be a collection of documents. The retrieval component selects a subset of relevant documents $D_k \subset D$ based on a similarity function:

$$
D_k = \text{Retriever}(x, D)
$$

$\bullet$ The generator then produces an output $y$ conditioned on both the input query and the retrieved documents:

$$
y \sim P(y \mid x, D_k)
$$

$\bullet$ The retriever uses the query to fetch relevant documents first, and only then the LLM generates the answer using query and those documents.  
$\bullet$ The overall pipeline can be summarized as:  

$$
x \rightarrow \text{Retriever} \rightarrow (x,D_k) \rightarrow \text{Generator} \rightarrow y
$$

$\bullet$ Advantages of RAG:  
$\qquad \star$ Access to external and up-to-date knowledge.  
$\qquad \star$ Reduced need for frequent model retraining.  
$\qquad \star$ Improved interpretability through retrieved evidence.  
$\bullet$ Limitations of RAG:  
$\qquad \star$ Performance depends on retrieval quality.  
$\qquad \star$ Increased system complexity and latency.

$\bullet$ Why can't we fine tune an LLM model with our private data (Ex: Company policies, HR data, Finance details)?  
$\qquad \blacksquare$ It takes a higher computational cost (needs to fine tune billions of parameters)  
$\qquad \blacksquare$ Need to fine tune the model continuously (as data getting updated all the time)

$\bullet$ Data Ingestion Pipeline - The way of storing all the external documents and data as vectors in a vector storage
$$
\textrm{Data(PDF, HTML, Excel, SQL DBs)} \rightarrow \textrm{Parsing(Chunks)} \rightarrow \textrm{Embeddings(Text→Vectors)} \rightarrow \textrm{Vector DB(Vector Storage)}
$$

$\bullet$ Retrieval Pipeline of a Traditional RAG - The way of a query becomes a vector and then finds similar data inside the vector storage using similarity search functions and then going to the LLM with a prompt
$$
\textrm{Query} \rightarrow \textrm{Embeddings(Text→Vectors)} \rightarrow \textrm{VectorDB(Vector Storage)} \rightarrow \textrm{Context} \rightarrow \textrm{Augmentation(Context + Prompt)} \rightarrow \textrm{LLM}
$$

In [13]:
# importing essential libraries
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### Document Structure
$\bullet$ Document structure contain two main parts,  
$\qquad \blacksquare$ Page Content  
$\qquad \blacksquare$ Meta Data

Here, we are loading one text file as a Langchain Document object

In [6]:
loader = TextLoader("Data/TextFiles/introduction.txt")
document = loader.load()
document

[Document(metadata={'source': 'Data/TextFiles/introduction.txt'}, page_content='Dilan Senuruk the first son of Vortex the Great in kingdom of Aerathis has invented the first chain reaction simulator of the sun. Dilan Senuruk himself is known as the Great sun god Nika which is a name given to the best scientific creation. He was married to the beauty queen Maleesha Ruvindi of Galanthrea and he is also a father to 3 children.\n\nThe First chain reaction simulator of the sun was made in 1967 and made public in 1971 so that every scientist in the continent can use it for research purposes. The chain reaction simulator works in fore steps and they are,\n1. The starting phase - Dummy photons and electrons are fed into the main valve\n2. Radiation phase - The top window opens for the sun radiation to fall on the main core\n3. Generating phase - The photos and electrons will combine to create bronton with the sun radiation\n4. Graphing phase - The bronton will fall on to the glass illuminating

Here, we are loading a whole directory with four PDFs as Langchain Document objects

In [7]:
dir_loader = DirectoryLoader("Data/PDFs", glob="**/*.pdf", loader_cls=PyMuPDFLoader, show_progress=False)
pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T10:48:43+05:30', 'source': 'Data\\PDFs\\Findings.pdf', 'file_path': 'Data\\PDFs\\Findings.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Dilan Senuruk', 'subject': '', 'keywords': '', 'moddate': '2026-04-17T10:48:43+05:30', 'trapped': '', 'modDate': "D:20260417104843+05'30'", 'creationDate': "D:20260417104843+05'30'", 'page': 0}, page_content='Findings of Vortex the Great \nKing Vortex the Great’s obsession with science began as a quiet curiosity but evolved into a \nrelentless pursuit that reshaped the intellectual foundations of his kingdom. Unlike traditional \nrulers who relied on inherited knowledge, Vortex established vast observatories and laboratories \nsuspended between the floating isles of Aerathis, where he could study the behavior of storms at \ntheir source. His earliest breakthrough came with the formulation of what he called Aero-\

In [9]:
# Read all the PDFs inside the directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"    ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"    ✗ Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents = process_all_pdfs("data")


found 4 PDF files to process

Processing: Findings.pdf
    ✓ Loaded 1 pages

Processing: Intro.pdf
    ✓ Loaded 1 pages

Processing: Kingdom_Expansion.pdf
    ✓ Loaded 1 pages

Processing: Sons.pdf
    ✓ Loaded 2 pages

Total documents loaded: 5


### Chunking
Split documents into smaller chunks for better RAG performance

In [16]:
# Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show exxample of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunks = split_documents(all_pdf_documents)

Split 5 documents into 16 chunks

Example chunk:
Content: Findings of Vortex the Great 
King Vortex the Great’s obsession with science began as a quiet curiosity but evolved into a 
relentless pursuit that reshaped the intellectual foundations of his kingdom...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T10:48:43+05:30', 'author': 'Dilan Senuruk', 'moddate': '2026-04-17T10:48:43+05:30', 'source': 'data\\PDFs\\Findings.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Findings.pdf', 'file_type': 'pdf'}
